# Módulo 14 · Semana 14, Selección de características
Juan Esteban Camacho Castro

Juan Diego Serpa Serpa

## Crea tu notebook y carga el dataset con la variable objetivo

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import VarianceThreshold
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestClassifier

In [2]:
df = pd.read_csv("S13_PD_Camacho_Serpa_novamarket_con_features.csv")

In [3]:
df["fuga_cliente"].value_counts()

fuga_cliente
0    473742
1    125258
Name: count, dtype: int64

## Separa entrenamiento y prueba antes de seleccionar nada

In [4]:
# Separar X e y
X = df.drop(columns=['fuga_cliente'])
y = df['fuga_cliente']

# Identificar y eliminar columnas de texto
columnas_texto = X.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

print("Columnas de texto eliminadas:")
print(columnas_texto)

X = X.drop(columns=columnas_texto)

Columnas de texto eliminadas:
['id_pedido', 'id_cliente', 'fecha_pedido', 'canal_compra', 'metodo_pago', 'ciudad_tienda', 'categoria_producto', 'producto', 'correo_cliente', 'nivel_satisfaccion', 'nivel_lealtad', 'comentario_cliente', 'fecha_actualizacion_stock', 'codigo_postal_agrupado', 'dia_semana_pedido', 'grupo_edad', 'proveedor_principal', 'bodega_asignada', 'ciudad_residencia', 'fecha_inscripcion_club', 'segmento_crm', 'acepta_promociones', 'dominio_correo', 'segmento_rfm']


In [5]:
X_entrenamiento, X_prueba, y_entrenamiento, y_prueba = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

### Corroboramos la longitud del conjunto de entrenamiento y de prueba para verificaar que no se haya quedado nada

In [6]:
len(X)

599000

In [7]:
len(X_entrenamiento) + len(X_prueba)

599000

### Corroboramos la proporción de los datos en el conjunto de prueba y de entrenamiento para la variable y

In [8]:
y_prueba.value_counts(normalize=True)

fuga_cliente
0    0.790885
1    0.209115
Name: proportion, dtype: float64

In [9]:
y_entrenamiento.value_counts(normalize=True)

fuga_cliente
0    0.790889
1    0.209111
Name: proportion, dtype: float64

## Filtro, descarta varianza casi nula y detecta redundancia

In [10]:
selector_varianza = VarianceThreshold(threshold=0.01)
selector_varianza.fit(X_entrenamiento)
columnas_con_varianza = X_entrenamiento.columns[selector_varianza.get_support()]

In [11]:
X_entrenamiento.corr()

,codigo_postal,precio_unitario,unidades_vendidas,monto_compra,edad_cliente,peso_pedido_kg,nivel_satisfaccion_imputado,ciudad_tienda_imputado,correo_cliente_imputado,comentario_cliente_imputado,...,antiguedad_cliente_dias,es_cliente_club,numero_compras,total_gastado,gasto_promedio,dias_desde_ultima_compra,score_R,score_F,score_M,rfm_score
codigo_postal,1.000000,-0.000891,0.000077,-0.001105,0.000851,0.002740,-0.001053,0.000675,0.000763,0.000509,...,0.000555,-0.000087,0.002146,0.003686,0.002711,-0.000823,0.002215,0.002725,0.003521,0.003626
precio_unitario,-0.000891,1.000000,0.001008,0.611210,0.000524,0.000026,-0.002578,-0.000888,-0.000399,0.001796,...,0.001743,-0.000090,0.000680,0.192981,0.308130,0.001224,-0.000408,-0.000201,0.194772,0.084743
unidades_vendidas,0.000077,0.001008,1.000000,0.402639,0.000633,0.000134,-0.001289,0.000751,0.001336,-0.001769,...,0.001886,-0.000578,0.000015,0.126430,0.201694,0.001257,-0.000691,-0.000190,0.119988,0.052011
monto_compra,-0.001105,0.611210,0.402639,1.000000,0.001260,-0.000894,-0.001246,-0.001087,-0.000287,0.000623,...,0.003681,-0.000426,-0.000026,0.315287,0.503498,0.001980,-0.000703,-0.000375,0.313573,0.136440
edad_cliente,0.000851,0.000524,0.000633,0.001260,1.000000,0.000062,0.000067,0.002133,0.001667,0.000222,...,0.004180,-0.001158,0.001124,0.001111,0.001187,-0.001679,0.000268,0.001805,0.001792,0.001620
peso_pedido_kg,0.002740,0.000026,0.000134,-0.000894,0.000062,1.000000,0.002284,0.003015,-0.000176,-0.000919,...,-0.000236,0.000557,0.001363,0.000630,-0.000747,-0.003070,0.002371,0.001255,0.000172,0.001597
nivel_satisfaccion_imputado,-0.001053,-0.002578,-0.001289,-0.001246,0.000067,0.002284,1.000000,0.003650,0.000055,0.003516,...,-0.000104,-0.000549,0.002267,0.000719,-0.000197,-0.000505,-0.000253,0.001735,0.001078,0.001111
ciudad_tienda_imputado,0.000675,-0.000888,0.000751,-0.001087,0.002133,0.003015,0.003650,1.000000,-0.001142,0.000115,...,0.000225,-0.000177,-0.002292,-0.002356,-0.000727,-0.000208,-0.000883,-0.001754,-0.002050,-0.002071
correo_cliente_imputado,0.000763,-0.000399,0.001336,-0.000287,0.001667,-0.000176,0.000055,-0.001142,1.000000,-0.001149,...,NaN,-0.624016,0.000444,-0.000718,-0.001474,-0.000198,0.000651,0.001082,-0.001004,0.000415
comentario_cliente_imputado,0.000509,0.001796,-0.001769,0.000623,0.000222,-0.000919,0.003516,0.000115,-0.001149,1.000000,...,-0.004551,-0.000760,0.000456,0.000353,0.000627,0.000414,0.001086,0.001703,0.001265,0.001753


## Filtro, calcula la correlación de cada feature con `fuga_cliente`

In [12]:
correlaciones_con_objetivo = (
    X_entrenamiento
    .corrwith(y_entrenamiento)
    .abs()
    .dropna()
    .sort_values(ascending=False)
)

print(correlaciones_con_objetivo)

C:\Users\puyol\anaconda3\envs\NovaMarket\Lib\site-packages\numpy\lib\_function_base_impl.py:3036: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\puyol\anaconda3\envs\NovaMarket\Lib\site-packages\numpy\lib\_function_base_impl.py:3037: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


nivel_lealtad_cod              0.218704
monto_compra_boxcox            0.103765
monto_compra                   0.087083
precio_unitario                0.068452
gasto_por_unidad               0.061437
canal_App                      0.047664
edad_cliente                   0.047426
gasto_promedio                 0.044348
es_venta_corporativa           0.033105
score_M                        0.028156
total_gastado                  0.026659
unidades_vendidas              0.020770
canal_Web                      0.016618
canal_Tienda                   0.016083
canal_Marketplace              0.014940
nivel_satisfaccion_imputado    0.012914
rfm_score                      0.011524
es_precio_alto                 0.007766
cat_Hogar                      0.002309
antiguedad_cliente_dias        0.002169
correo_cliente_imputado        0.002012
score_R                        0.001588
num_palabras_comentario        0.001581
numero_compras                 0.001552
longitud_comentario            0.001408


## Wrapper, aplica RFE con regresión logística

Todavia hay datos faltantes en algunas columnas así que en vez de eliminarlas completamente vamos a utilizar SimpleImputer

In [13]:
escalador = StandardScaler()
# Imputar valores faltantes
imputador = SimpleImputer(strategy="median")

X_entrenamiento = pd.DataFrame(
    imputador.fit_transform(X_entrenamiento),
    columns=X_entrenamiento.columns,
    index=X_entrenamiento.index
)

X_prueba = pd.DataFrame(
    imputador.transform(X_prueba),
    columns=X_prueba.columns,
    index=X_prueba.index
)

# Estandarizar las variables
escalador = StandardScaler()

X_entrenamiento_escalado = pd.DataFrame(
    escalador.fit_transform(X_entrenamiento),
    columns=X_entrenamiento.columns,
    index=X_entrenamiento.index
)

X_prueba_escalado = pd.DataFrame(
    escalador.transform(X_prueba),
    columns=X_prueba.columns,
    index=X_prueba.index
)

# Aplicar RFE
modelo_base = LogisticRegression(max_iter=5000)

selector_rfe = RFE(
    estimator=modelo_base,
    n_features_to_select=8
)

selector_rfe.fit(X_entrenamiento_escalado, y_entrenamiento)

features_elegidas_rfe = X_entrenamiento.columns[
    selector_rfe.support_
]

print("Variables seleccionadas por RFE:")
print(features_elegidas_rfe.tolist())

Variables seleccionadas por RFE:
['precio_unitario', 'monto_compra', 'edad_cliente', 'es_venta_corporativa', 'canal_App', 'canal_Web', 'nivel_lealtad_cod', 'monto_compra_boxcox']


In [14]:
 X_entrenamiento.columns [selector_rfe.ranking_ == 1]

Index(['precio_unitario', 'monto_compra', 'edad_cliente',
       'es_venta_corporativa', 'canal_App', 'canal_Web', 'nivel_lealtad_cod',
       'monto_compra_boxcox'],
      dtype='str')

## Embedded, entrena un Random Forest y extrae las importancias

### Regularización L1

In [15]:
modelo_lasso = Lasso(alpha=0.05)
modelo_lasso.fit(X_entrenamiento, y_entrenamiento)

features_lasso = X_entrenamiento.columns[modelo_lasso.coef_ != 0]

### Random Forest

In [16]:
modelo_bosque = RandomForestClassifier(n_estimators=200, random_state=42)
modelo_bosque.fit(X_entrenamiento, y_entrenamiento)

importancias = pd.Series(
    modelo_bosque.feature_importances_, index=X_entrenamiento.columns
).sort_values(ascending=False)

In [17]:
print(importancias)

gasto_promedio                 0.052317
total_gastado                  0.050450
nivel_lealtad_cod              0.049754
dias_desde_ultima_compra       0.049216
dias_desde_pedido              0.049011
codigo_postal                  0.046968
precio_unitario                0.046108
peso_pedido_kg                 0.045672
edad_cliente                   0.044974
monto_compra_boxcox            0.042693
puntos_fidelidad_actuales      0.042606
antiguedad_cliente_dias        0.042305
monto_compra                   0.041697
gasto_por_unidad               0.040635
ciudad_tienda_cod              0.029179
z_monto_compra                 0.026843
longitud_comentario            0.026383
mes_pedido                     0.025615
numero_compras                 0.021719
rfm_score                      0.020715
dias_reposicion                0.014357
stock_minimo                   0.014336
num_palabras_comentario        0.014329
trimestre_pedido               0.012086
score_R                        0.011892


## Tabla comparativa de los tres métodos

Armamos una tabla con una fila por cada feature candidata y tres columnas booleanas que indican si cada método la seleccionó: filtro (top 10 por correlación con `fuga_cliente`), wrapper RFE y embedded (top 8 por importancia de Random Forest).

In [18]:
# Features seleccionadas por el filtro (top 10 por correlación con el objetivo)
top_filtro = correlaciones_con_objetivo.head(10).index.tolist()
print("Top 10 por filtro de correlación:")
print(top_filtro)

Top 10 por filtro de correlación:
['nivel_lealtad_cod', 'monto_compra_boxcox', 'monto_compra', 'precio_unitario', 'gasto_por_unidad', 'canal_App', 'edad_cliente', 'gasto_promedio', 'es_venta_corporativa', 'score_M']


In [19]:
# Features seleccionadas por RFE
top_rfe = features_elegidas_rfe.tolist()
print("Features por RFE:")
print(top_rfe)

Features por RFE:
['precio_unitario', 'monto_compra', 'edad_cliente', 'es_venta_corporativa', 'canal_App', 'canal_Web', 'nivel_lealtad_cod', 'monto_compra_boxcox']


In [20]:
# Top 8 por importancia de Random Forest
top_rf = importancias.head(8).index.tolist()
print("Top 8 por Random Forest:")
print(top_rf)

Top 8 por Random Forest:
['gasto_promedio', 'total_gastado', 'nivel_lealtad_cod', 'dias_desde_ultima_compra', 'dias_desde_pedido', 'codigo_postal', 'precio_unitario', 'peso_pedido_kg']


In [21]:
# Unión de todas las features candidatas
todas_features = sorted(set(top_filtro) | set(top_rfe) | set(top_rf))

tabla_comparativa = pd.DataFrame({"feature": todas_features})
tabla_comparativa["filtro"] = tabla_comparativa["feature"].isin(top_filtro)
tabla_comparativa["wrapper_rfe"] = tabla_comparativa["feature"].isin(top_rfe)
tabla_comparativa["embedded_rf"] = tabla_comparativa["feature"].isin(top_rf)

print(tabla_comparativa)

                     feature  filtro  wrapper_rfe  embedded_rf
0                  canal_App    True         True        False
1                  canal_Web   False         True        False
2              codigo_postal   False        False         True
3          dias_desde_pedido   False        False         True
4   dias_desde_ultima_compra   False        False         True
5               edad_cliente    True         True        False
6       es_venta_corporativa    True         True        False
7           gasto_por_unidad    True        False        False
8             gasto_promedio    True        False         True
9               monto_compra    True         True        False
10       monto_compra_boxcox    True         True        False
11         nivel_lealtad_cod    True         True         True
12            peso_pedido_kg   False        False         True
13           precio_unitario    True         True         True
14                   score_M    True        False      

## Veces seleccionada por cada feature

In [22]:
tabla_comparativa["veces_seleccionada"] = tabla_comparativa[["filtro", "wrapper_rfe", "embedded_rf"]].sum(axis=1)
tabla_comparativa = tabla_comparativa.sort_values("veces_seleccionada", ascending=False).reset_index(drop=True)
print(tabla_comparativa)

                     feature  filtro  wrapper_rfe  embedded_rf  \
0            precio_unitario    True         True         True   
1          nivel_lealtad_cod    True         True         True   
2                  canal_App    True         True        False   
3               edad_cliente    True         True        False   
4             gasto_promedio    True        False         True   
5               monto_compra    True         True        False   
6       es_venta_corporativa    True         True        False   
7        monto_compra_boxcox    True         True        False   
8           gasto_por_unidad    True        False        False   
9   dias_desde_ultima_compra   False        False         True   
10             codigo_postal   False        False         True   
11         dias_desde_pedido   False        False         True   
12                 canal_Web   False         True        False   
13            peso_pedido_kg   False        False         True   
14        

## Subconjunto final de features
Priorizamos las features con `veces_seleccionada` igual a 3 o 2, revisando que no haya pares redundantes (correlación > 0.85) entre ellas. 

Dentro del grupo de 2 votos tenemos `monto_compra` y `monto_compra_boxcox`, que son redundantes (uno es transformación directa del otro). Nos quedamos solo con `monto_compra` por ser más interpretable.

Agregamos `dias_desde_ultima_compra` del grupo de 1 voto porque aporta una señal nueva de recencia que ninguna otra feature cubre, y la recencia es una señal clave para predecir fuga de clientes.

In [23]:
features_finales = [
    "nivel_lealtad_cod",
    "canal_App",
    "es_venta_corporativa",
    "edad_cliente",
    "gasto_promedio",
    "precio_unitario",
    "monto_compra",
    "dias_desde_ultima_compra"
]
print("Total features finales:", len(features_finales))
print(features_finales)

Total features finales: 8
['nivel_lealtad_cod', 'canal_App', 'es_venta_corporativa', 'edad_cliente', 'gasto_promedio', 'precio_unitario', 'monto_compra', 'dias_desde_ultima_compra']


### Verificación: ningún par redundante en la selección final

In [24]:
correlaciones_finales = X_entrenamiento[features_finales].corr().abs()
print(correlaciones_finales)

                          nivel_lealtad_cod  canal_App  es_venta_corporativa  \
nivel_lealtad_cod                  1.000000   0.001737              0.000130   
canal_App                          0.001737   1.000000              0.000302   
es_venta_corporativa               0.000130   0.000302              1.000000   
edad_cliente                       0.001645   0.001213              0.000812   
gasto_promedio                     0.001634   0.000103              0.345624   
precio_unitario                    0.001135   0.000522              0.329915   
monto_compra                       0.000624   0.000607              0.685914   
dias_desde_ultima_compra           0.000916   0.000301              0.001046   

                          edad_cliente  gasto_promedio  precio_unitario  \
nivel_lealtad_cod             0.001645        0.001634         0.001135   
canal_App                     0.001213        0.000103         0.000522   
es_venta_corporativa          0.000812        0.345624

In [25]:
# Buscar pares con correlación > 0.85
import numpy as np
mascara = np.triu(np.ones(correlaciones_finales.shape), k=1).astype(bool)
pares_redundantes = correlaciones_finales.where(mascara).stack()
pares_redundantes = pares_redundantes[pares_redundantes > 0.85]
print("Pares redundantes (correlación > 0.85):")
print(pares_redundantes if len(pares_redundantes) > 0 else "Ninguno")

Pares redundantes (correlación > 0.85):
Ninguno


## Exportación del dataset con las features seleccionadas

In [26]:
df_final = df[features_finales + ["fuga_cliente", "id_cliente"]]
df_final.shape

(599000, 10)

In [27]:
df_final.to_csv("novamarket_features_seleccionadas.csv", index=False)